# Silver: Cross-Source Job Identity Mapping

Detects and links duplicate jobs across different sources.

## Architecture

**Input**: `silver.silver_jobs_current`  
**Output**: `silver.silver_job_identity_map`  
**Mode**: Incremental (process new batches only)

## Batch Processing

- Tracks processed batches in `metadata.job_identity_batches`
- Automatically processes all unprocessed batches
- Idempotent: safe to re-run
- Creates identity mappings for cross-source duplicates

## Matching Strategy (Hierarchical)

1. **Exact match** (confidence: 1.0):
   - Same company + title + location
   
2. **Fuzzy company match** (confidence: 0.85):
   - Similar company names (normalized) + title + location
   
3. **URL match** (confidence: 0.95):
   - Same posting URL across sources
   
4. **Composite hash** (confidence: 0.80):
   - Company + title + description snippet hash

## Output

- Creates `silver.silver_job_identity_map`
- Links source jobs to canonical enterprise_job_id
- Enables cross-source analytics and deduplication

In [0]:
dbutils.widgets.text("batch_id", "", "Batch ID (leave empty for all unprocessed)")
dbutils.widgets.text("match_threshold", "0.75", "Minimum match score for linking")

batch_id = dbutils.widgets.get("batch_id").strip()
match_threshold = float(dbutils.widgets.get("match_threshold"))

In [0]:
import hashlib
import json
from datetime import datetime
from pyspark.sql import functions as F, Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType, LongType, DoubleType
from difflib import SequenceMatcher

CATALOG = "workspace"
SILVER_SCHEMA = f"{CATALOG}.silver"
METADATA_SCHEMA = f"{CATALOG}.metadata"

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
run_timestamp_py = datetime.now()  # Python timestamp for metadata records
run_timestamp = F.current_timestamp()  # Spark timestamp for DataFrame operations

print(f"Run ID: {run_id}")

In [0]:
# Create metadata table to track identity-mapped batches
metadata_table = f"{METADATA_SCHEMA}.job_identity_batches"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {metadata_table} (
  batch_id STRING,
  source_name STRING,
  jobs_processed INT,
  exact_matches INT,
  fuzzy_matches INT,
  hash_matches INT,
  processed_at TIMESTAMP,
  run_id STRING,
  status STRING
)
USING DELTA
COMMENT 'Tracks which batches have been identity-mapped'
""")

# Define metadata schema for recording batch processing
metadata_schema = StructType([
    StructField("batch_id", StringType(), True),
    StructField("source_name", StringType(), True),
    StructField("jobs_processed", IntegerType(), True),
    StructField("exact_matches", IntegerType(), True),
    StructField("fuzzy_matches", IntegerType(), True),
    StructField("hash_matches", IntegerType(), True),
    StructField("processed_at", TimestampType(), True),
    StructField("run_id", StringType(), True),
    StructField("status", StringType(), True)
])

In [0]:
# Identify unprocessed batches
if batch_id:
    # Process specific batch if provided
    batches_to_process = [(batch_id, None)]  # (batch_id, source_name) tuple
else:
    # Find all batches in current table
    all_batches_query = f"""
        SELECT DISTINCT current_batch_id as batch_id, source_name
        FROM {SILVER_SCHEMA}.silver_jobs_current
        WHERE is_active = true AND soft_delete_flag = false
        AND current_batch_id IS NOT NULL
        AND current_batch_id != ''
        AND source_name IS NOT NULL
    """
    
    all_batches_df = spark.sql(all_batches_query)
    
    # Find already processed batches
    processed_batches_df = spark.sql(f"""
        SELECT DISTINCT batch_id, source_name
        FROM {metadata_table}
        WHERE status = 'success'
    """)
    
    # Find unprocessed batches (in current but not in metadata)
    unprocessed_df = all_batches_df.join(
        processed_batches_df,
        on=["batch_id", "source_name"],
        how="left_anti"
    ).orderBy("batch_id", "source_name")
    
    batches_to_process = [(row.batch_id, row.source_name) for row in unprocessed_df.collect()]
    
    if not batches_to_process:
        dbutils.notebook.exit({"status": "success", "message": "No unprocessed batches", "total_matches": 0})
    
    print(f"Found {len(batches_to_process)} unprocessed batch(es) to process")

In [0]:
def string_similarity(s1, s2):
    """Calculate string similarity using SequenceMatcher"""
    if not s1 or not s2:
        return 0.0
    return SequenceMatcher(None, s1.lower(), s2.lower()).ratio()

def generate_composite_hash(company, title, description):
    """Generate composite hash for job matching"""
    # Use first 500 chars of description
    desc_snippet = (description or "")[:500]
    composite = f"{company}|{title}|{desc_snippet}".lower()
    return hashlib.md5(composite.encode()).hexdigest()

# Register UDFs
similarity_udf = F.udf(string_similarity, DoubleType())
composite_hash_udf = F.udf(generate_composite_hash, StringType())

print("Matching functions registered")

In [0]:
# Initialize tracking
total_jobs_processed = 0
total_exact_matches = 0
total_fuzzy_matches = 0
total_hash_matches = 0
processed_batch_count = 0
failed_batches = []

# Ensure identity map table exists
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER_SCHEMA}.silver_job_identity_map (
  job_identity_map_sk BIGINT,
  job_id_1 STRING,
  job_id_2 STRING,
  source_1 STRING,
  source_2 STRING,
  match_rule STRING,
  match_score DECIMAL(5,4),
  assigned_at TIMESTAMP,
  batch_id STRING
)
USING DELTA
""")

# Load ALL active jobs for comparison (needed for cross-source matching)
all_jobs_df = spark.sql(f"""
SELECT 
    enterprise_job_id,
    source_name,
    source_job_id,
    source_job_key,
    company_name_norm,
    title_normalized,
    location_norm,
    source_url,
    description_raw,
    record_hash,
    current_batch_id
FROM {SILVER_SCHEMA}.silver_jobs_current
WHERE is_active = true AND soft_delete_flag = false
""")

# Add matching keys to all jobs for efficient comparison
jobs_with_keys_df = all_jobs_df.withColumn(
    "exact_match_key",
    F.concat_ws("|", 
        F.coalesce(F.col("company_name_norm"), F.lit("")),
        F.coalesce(F.col("title_normalized"), F.lit("")),
        F.coalesce(F.col("location_norm"), F.lit(""))
    )
).withColumn(
    "company_title_key",
    F.concat_ws("|",
        F.coalesce(F.col("company_name_norm"), F.lit("")),
        F.coalesce(F.col("title_normalized"), F.lit(""))
    )
).withColumn(
    "composite_hash",
    F.sha2(
        F.concat_ws("|",
            F.coalesce(F.col("company_name_norm"), F.lit("")),
            F.coalesce(F.col("title_normalized"), F.lit("")),
            F.substring(F.coalesce(F.col("description_raw"), F.lit("")), 1, 500)
        ), 256
    )
)

print(f"Loaded {all_jobs_df.count()} jobs with matching keys generated")

In [0]:
# Process each batch (jobs_with_keys_df already created and cached in previous cell)
for current_batch_id, current_source in batches_to_process:
    try:
        print(f"Finding identity matches for batch {current_batch_id} ({current_source})...", end=" ")
        
        # Get jobs in current batch
        batch_jobs_df = jobs_with_keys_df.filter(
            (F.col("current_batch_id") == current_batch_id)
        )
        
        if current_source:
            batch_jobs_df = batch_jobs_df.filter(F.col("source_name") == current_source)
        
        job_count = batch_jobs_df.count()
        
        if job_count == 0:
            print("No jobs found")
            continue
        
        # Find exact matches (same company + title + location, different sources)
        exact_matches_df = batch_jobs_df.alias("j1").join(
            jobs_with_keys_df.alias("j2"),
            (
                (F.col("j1.exact_match_key") == F.col("j2.exact_match_key")) &
                (F.col("j1.source_name") != F.col("j2.source_name")) &
                (F.col("j1.source_job_key") < F.col("j2.source_job_key"))
            ),
            "inner"
        ).select(
            F.col("j1.enterprise_job_id").alias("job_id_1"),
            F.col("j2.enterprise_job_id").alias("job_id_2"),
            F.col("j1.source_name").alias("source_1"),
            F.col("j2.source_name").alias("source_2"),
            F.lit("EXACT_MATCH").alias("match_rule"),
            F.lit(1.0).alias("match_score")
        )
        
        exact_count = exact_matches_df.count()
        
        # Find fuzzy company matches (similar company + same title, different sources)
        fuzzy_matches_df = batch_jobs_df.alias("j1").join(
            jobs_with_keys_df.alias("j2"),
            (
                (F.col("j1.company_title_key") == F.col("j2.company_title_key")) &
                (F.col("j1.source_name") != F.col("j2.source_name")) &
                (F.col("j1.source_job_key") < F.col("j2.source_job_key"))
            ),
            "inner"
        ).select(
            F.col("j1.enterprise_job_id").alias("job_id_1"),
            F.col("j2.enterprise_job_id").alias("job_id_2"),
            F.col("j1.source_name").alias("source_1"),
            F.col("j2.source_name").alias("source_2"),
            F.lit("FUZZY_COMPANY").alias("match_rule"),
            F.lit(0.85).alias("match_score")
        )
        
        fuzzy_count = fuzzy_matches_df.count()
        
        # Find composite hash matches
        hash_matches_df = batch_jobs_df.alias("j1").join(
            jobs_with_keys_df.alias("j2"),
            (
                (F.col("j1.composite_hash") == F.col("j2.composite_hash")) &
                (F.col("j1.source_name") != F.col("j2.source_name")) &
                (F.col("j1.source_job_key") < F.col("j2.source_job_key"))
            ),
            "inner"
        ).select(
            F.col("j1.enterprise_job_id").alias("job_id_1"),
            F.col("j2.enterprise_job_id").alias("job_id_2"),
            F.col("j1.source_name").alias("source_1"),
            F.col("j2.source_name").alias("source_2"),
            F.lit("COMPOSITE_HASH").alias("match_rule"),
            F.lit(0.95).alias("match_score")
        )
        
        hash_count = hash_matches_df.count()
        
        # Union all match types
        all_matches_df = exact_matches_df.union(fuzzy_matches_df).union(hash_matches_df)
        total_matches = all_matches_df.count()
        
        # Write matches if found
        if total_matches > 0:
            # Add metadata
            matches_final_df = all_matches_df.withColumn(
                "job_identity_map_sk", F.monotonically_increasing_id()
            ).withColumn(
                "assigned_at", run_timestamp
            ).withColumn(
                "batch_id", F.lit(current_batch_id)
            )
            
            # Write to identity map table
            matches_final_df.select(
                "job_identity_map_sk",
                "job_id_1",
                "job_id_2",
                "source_1",
                "source_2",
                "match_rule",
                F.col("match_score").cast("decimal(5,4)"),
                "assigned_at",
                "batch_id"
            ).write.format("delta").mode("append").saveAsTable(f"{SILVER_SCHEMA}.silver_job_identity_map")
        
        # Record batch processing in metadata
        safe_source = current_source if current_source else "unknown"
        metadata_record = spark.createDataFrame([
            (current_batch_id, safe_source, int(job_count), int(exact_count), int(fuzzy_count), int(hash_count), run_timestamp_py, run_id, "success")
        ], schema=metadata_schema)
        
        metadata_record.write \
            .format("delta") \
            .mode("append") \
            .saveAsTable(metadata_table)
        
        # Update tracking
        total_jobs_processed += job_count
        total_exact_matches += exact_count
        total_fuzzy_matches += fuzzy_count
        total_hash_matches += hash_count
        processed_batch_count += 1
        
        print(f"✓ Jobs:{job_count} Exact:{exact_count} Fuzzy:{fuzzy_count} Hash:{hash_count}")
        
    except Exception as e:
        print(f"✗ {str(e)}")
        failed_batches.append((current_batch_id, current_source, str(e)))
        
        # Record failure in metadata
        try:
            safe_source = current_source if current_source else "unknown"
            failure_record = spark.createDataFrame([
                (current_batch_id, safe_source, 0, 0, 0, 0, run_timestamp_py, run_id, f"failed: {str(e)}")
            ], schema=metadata_schema)
            
            failure_record.write \
                .format("delta") \
                .mode("append") \
                .saveAsTable(metadata_table)
        except:
            pass
        
        continue

print(f"\nProcessed {processed_batch_count} batches, {total_jobs_processed} jobs, found {total_exact_matches + total_fuzzy_matches + total_hash_matches} total matches")
if failed_batches:
    print(f"Failed: {len(failed_batches)} batches")

In [0]:
# Show identity mapping history
if processed_batch_count > 0:
    metadata_df = spark.sql(f"""
        SELECT batch_id, source_name, jobs_processed, exact_matches, fuzzy_matches, hash_matches, processed_at, status
        FROM {metadata_table}
        ORDER BY processed_at DESC
        LIMIT 20
    """)
    display(metadata_df)
    
    # Show current identity map summary
    identity_summary_df = spark.sql(f"""
        SELECT 
            match_rule_name,
            COUNT(*) as match_count,
            ROUND(AVG(match_score), 3) as avg_score
        FROM {SILVER_SCHEMA}.silver_job_identity_map
        GROUP BY match_rule_name
        ORDER BY match_count DESC
    """)
    display(identity_summary_df)
    
    # Show sample matches
    sample_matches_df = spark.sql(f"""
        SELECT enterprise_job_id, source_name, source_job_id, match_rule_name, match_score, assigned_at
        FROM {SILVER_SCHEMA}.silver_job_identity_map
        ORDER BY assigned_at DESC
        LIMIT 10
    """)
    display(sample_matches_df)

# Exit summary
result = {
    "status": "success" if len(failed_batches) == 0 else "partial_success",
    "run_id": run_id,
    "batches_processed": processed_batch_count,
    "batches_failed": len(failed_batches),
    "records_processed": total_jobs_processed,  # Standardized field name for workflow integration
    "jobs_processed": total_jobs_processed,  # Legacy field name for backward compatibility
    "exact_matches": total_exact_matches,
    "fuzzy_matches": total_fuzzy_matches,
    "hash_matches": total_hash_matches,
    "total_matches": total_exact_matches + total_fuzzy_matches + total_hash_matches,
    "metadata_table": metadata_table
}

dbutils.notebook.exit(json.dumps(result))